# Demo: MUSHRA-like 2D Experiment

To get started, download this repository navigate to the folder in which you downloaded it and
```bash
pip install -e .
```
If you did this, you can run the example below.

In [ ]:
import whispy
from pathlib import Path

## Define the experiment through configuration files

Config files are used to define an experiment. For modularity, multiple configs are used. They can be changed, to change the experiment.

In [ ]:
# directory containing the config files
config_dir = Path('..') / 'configs'

# This defines the attributes/qualities for the experiment.
config_attributes = config_dir / 'attributes.yml'

# This defines the stimuli used in the experiment
config_stimuli = config_dir / 'stimuli.yml'

# This defines the course of the experiment. The content here works hand in
# hand with the attributes and stimuli configs
config_experiment = config_dir / 'experiment.yml'

# This is the configuration of the rating GUI that is used in this example
config_drag_and_drop_mushra = config_dir / 'drag_and_drop_mushra.yml'


## Load the stimuli handler

Different 'Handlers' can be used to control stimuli playback. This example uses audio files stored in wav-files. This can be handled by the `SounddeviceHandler`

In [ ]:
# directory containing the audio files
stimuli_dir = Path('..') / 'stimuli'

# create the handler
stimuli_handler = whispy.interfaces.SoundDevice(
    config_stimuli, stimuli_dir, loop=True)

each handler has a `play` and `stop` method. It gets the stimulus name, which is defined in `config_stimuli`. In this case the stimulus names are `1`, `2` and `3`. Comment out an run the below to play around with this.

In [ ]:
# play stimulus 1
#stimuli_handler.play(1)

# stop stimulus 1
# stimuli_handler.stop(1)

## Schedule the experiment

An experiment often consists of multiple conditions and in addition multiple test might be performed or multiple attributes rated. This is defined in `config_experiment`.

In most cases, parts of the experiment are randomized and each participant of the experiments performs the tasks in a different order. This is done with the `ExperimentalScheduler` class.

It takes the experiment defined in `config_experiments`, randomizes the desired parts, and returns a class instance that can be iterated.

In [ ]:
# All parts of the experiment are randomized using the default parameter values
schedule = whispy.ExperimentScheduler(config_experiment)

The iteration shows, how the experiment will be structured

In [ ]:
for screen in schedule:
    print(screen)

## Run the experiment

A little coding is required to actually run the experiment. The advantage of this is, that everything stays very flexible and lots of different experiments can be designed.

In [ ]:
# initialize variable to save the results
results = None

for screen in schedule:

    # use screen meta data to display information to participant
    if screen["block_changed"]:
        whispy.gui.InfoWindow(f"Please rate the {screen['attribute']} next.")

    # start rating screen
    mushra_like = whispy.methods.DragAndDropMUSHRA(
        screen=screen,
        stimuli_handler=stimuli_handler,
        attributes=config_attributes,
        drag_and_drop_mushra=config_drag_and_drop_mushra)

    # update results
    results = mushra_like.get_results(results)

The results were updated iteratively. They contain the ratings in *long* format and are stored in a pandas DataFrame:

In [ ]:
print(results)